# Week 1 - Day 3: Model Evaluation + Improvement

Move from **"it works"** to **"I understand it like a pro"**: confusion matrix, precision/recall, and comparing models.

## Goal
- See why accuracy alone can mislead on imbalanced data
- Use **confusion matrix** to see true/false positives and negatives
- Read **precision, recall, F1** for spam vs ham
- Compare **Naive Bayes** vs **Logistic Regression**
- Use **`class_weight='balanced'`** to prioritize minority class (spam)

## Step 1: Imports and setup

In [1]:
import os
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

## Step 2: NLTK resources (same as Day 1–2)

In [2]:
LOCAL_NLTK_DIR = os.path.join(os.getcwd(), '.nltk_data')
os.makedirs(LOCAL_NLTK_DIR, exist_ok=True)
if LOCAL_NLTK_DIR not in nltk.data.path:
    nltk.data.path.insert(0, LOCAL_NLTK_DIR)

nltk.download('punkt', quiet=True, download_dir=LOCAL_NLTK_DIR)
nltk.download('punkt_tab', quiet=True, download_dir=LOCAL_NLTK_DIR)
nltk.download('stopwords', quiet=True, download_dir=LOCAL_NLTK_DIR)
print('NLTK ready')

NLTK ready


## Step 3: Load data, preprocess, TF-IDF (same pipeline as Day 2)

In [3]:
def clean_text(text: str):
    text = str(text).lower()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w.isalnum()]
    sw = set(stopwords.words('english'))
    return [w for w in tokens if w not in sw]

df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']].copy()
df.columns = ['label', 'text']
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

df['cleaned_text'] = df['text'].apply(clean_text)
df['cleaned_text_str'] = df['cleaned_text'].apply(lambda x: ' '.join(x))

tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['cleaned_text_str'])
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train/test shapes:', X_train.shape, X_test.shape)
print('Class balance (train):\n', y_train.value_counts())

Train/test shapes: (4457, 3000) (1115, 3000)
Class balance (train):
 label_num
0    3859
1     598
Name: count, dtype: int64


## Step 4: Why accuracy is not enough

With **imbalanced** data (more ham than spam), a model can score high accuracy by predicting ham most of the time. We need **confusion matrix** and **per-class** metrics.

## Step 5: Baseline — Multinomial Naive Bayes

In [4]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

print('Accuracy (Naive Bayes):', round(accuracy_score(y_test, y_pred_nb), 4))
print('\nConfusion matrix [rows=actual, cols=predicted]')
print('Labels: 0=ham, 1=spam')
print(confusion_matrix(y_test, y_pred_nb))
print('\nClassification report:')
print(classification_report(y_test, y_pred_nb, target_names=['ham', 'spam']))

Accuracy (Naive Bayes): 0.9749

Confusion matrix [rows=actual, cols=predicted]
Labels: 0=ham, 1=spam
[[965   1]
 [ 27 122]]

Classification report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       966
        spam       0.99      0.82      0.90       149

    accuracy                           0.97      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.98      0.97      0.97      1115



### Reading the confusion matrix

Example interpretation:
- Top-left: **true ham** predicted as ham
- Top-right: ham predicted as spam (false alarm)
- Bottom-left: **spam missed** (predicted as ham) — often costly
- Bottom-right: spam caught correctly

## Step 6: Logistic Regression (default)

In [5]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print('Accuracy (Logistic Regression):', round(accuracy_score(y_test, y_pred_lr), 4))
print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred_lr))
print('\nClassification report:')
print(classification_report(y_test, y_pred_lr, target_names=['ham', 'spam']))

Accuracy (Logistic Regression): 0.9641

Confusion matrix:
[[965   1]
 [ 39 110]]

Classification report:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       0.99      0.74      0.85       149

    accuracy                           0.96      1115
   macro avg       0.98      0.87      0.91      1115
weighted avg       0.97      0.96      0.96      1115



/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


## Step 7: Logistic Regression with balanced class weights

In [6]:
lr_balanced = LogisticRegression(
    max_iter=1000, random_state=42, class_weight='balanced'
)
lr_balanced.fit(X_train, y_train)
y_pred_bal = lr_balanced.predict(X_test)

print('Accuracy (LR balanced):', round(accuracy_score(y_test, y_pred_bal), 4))
print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred_bal))
print('\nClassification report:')
print(classification_report(y_test, y_pred_bal, target_names=['ham', 'spam']))

Accuracy (LR balanced): 0.9767

Confusion matrix:
[[952  14]
 [ 12 137]]

Classification report:
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.91      0.92      0.91       149

    accuracy                           0.98      1115
   macro avg       0.95      0.95      0.95      1115
weighted avg       0.98      0.98      0.98      1115



/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


`class_weight='balanced'` upweights the minority class so the model pays more attention to **spam** — often improves **spam recall** at some cost to ham precision.

## Step 8: Side-by-side comparison

In [7]:
def spam_metrics(y_true, y_pred):
    # label 1 = spam
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average=None, zero_division=0
    )
    return float(p[0]), float(r[0]), float(f1[0])

rows = []
for name, yp in [
    ('Naive Bayes', y_pred_nb),
    ('Logistic Regression', y_pred_lr),
    ('LR + class_weight=balanced', y_pred_bal),
]:
    acc = accuracy_score(y_test, yp)
    sp, sr, sf1 = spam_metrics(y_test, yp)
    rows.append({
        'model': name,
        'accuracy': round(acc, 4),
        'spam_precision': round(sp, 4),
        'spam_recall': round(sr, 4),
        'spam_f1': round(sf1, 4),
    })

comparison = pd.DataFrame(rows)
comparison

,model,accuracy,spam_precision,spam_recall,spam_f1
0,Naive Bayes,0.9749,0.9919,0.8188,0.8971
1,Logistic Regression,0.9641,0.9910,0.7383,0.8462
2,LR + class_weight=balanced,0.9767,0.9073,0.9195,0.9133


## Day 3 checklist
- [x] Confusion matrix interpreted
- [x] Precision / recall / F1 for spam understood
- [x] Compared Naive Bayes vs Logistic Regression
- [x] Tried imbalance handling via `class_weight`

### Next (Day 4)
Build a **Streamlit** app: input text → spam/ham + probability.